### Stream Customers Data from cloud files to Delta Lake
 
 1. Read files from cloud storage using DataStreamReader API
 2. Transform the dataframe to add the following columns\
        - file path: Cloud file path\
        - ingestion date: Current Time Stamp
 3. Write the transformed data stream to a Delta lake table

**Read files from cloud storage using DataStreamReader API**

In [0]:
from pyspark.sql.types import *

customer_schema=StructType(fields=[
StructField('customer_id',LongType(),True),
StructField('customer_name',StringType(),True),
StructField('date_of_birth',StringType(),True),
StructField('email',StringType(),True),
StructField('member_since',StringType(),True),
StructField('telephone',StringType(),True),
StructField('created_timestamp',StringType(),True)
])

In [0]:
dfCustomers=(spark.readStream
             .format('json')
             .schema(customer_schema)
             .load('/Volumes/gizmobox_gr/landing/operational_data/customer_stream')             
            )

**Write the transformed data stream to a Delta lake table**

In [0]:
dfStreamingQuery=(
    dfCustomers.writeStream
    .format('delta')
    .option("checkpointLocation", "/Volumes/gizmobox_gr/landing/operational_data/customers_stream/_checkpoints_stream")
    #.trigger(once=True, processingTime='10 seconds', maxFilesPerTrigger=1, maxBytesPerTrigger=10000)
    .outputMode('append', 'update')
    .toTable('gizmobox_gr.bronze.customers_stream')
)

In [0]:
dfStreamingQuery.stop()

In [0]:
display(spark.table('gizmobox_gr.bronze.customers_stream'))